In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS pep_dev.eval;

CREATE OR REPLACE TABLE pep_dev.eval.own_stmt AS 
SELECT entity_id, canonical_id, schema, prop, prop_type, dataset,value
FROM pep_dev.bronze.statements
WHERE _snapshot_version= (SELECT MAX(snapshot_version) FROM pep_dev.bronze.snapshot_version_control)
AND SCHEMA IN ('Ownership', 'Company', 'Organization', 'LegalEntity');


-- SELECT * FROM pep_dev.bronze.statements
-- LIMIT 10

In [0]:
-- SELECT * FROM pep_dev.eval.own_stmt;

SELECT prop , count(*) as n_rows, COUNT(DISTINCT entity_id) as n_entities,
ROUND (100*COUNT(DISTINCT entity_id)/t.total,1) as coverage_pct,t.total
FROM pep_dev.eval.own_stmt
CROSS JOIN (SELECT COUNT(DISTINCT entity_id) as total FROM pep_dev.eval.own_stmt
WHERE schema='Ownership') t 
WHERE SCHEMA ='Ownership'
GROUP BY prop,t.total
ORDER BY n_entities DESC


In [0]:
SELECT * FROM pep_dev.eval.own_stmt
LIMIT 100;

In [0]:
%sql

SELECT VALUE,count(*) FROM pep_dev.eval.own_stmt

WHERE prop = 'percentage'
 
GROUP BY value

ORDER BY value DESC

LIMIT 10000

In [0]:
%sql

SELECT try_cast(VALUE AS double),value,count(*) FROM pep_dev.eval.own_stmt

WHERE prop = 'percentage' AND try_cast(VALUE AS double) IS NULL 
 
GROUP BY value

ORDER BY value DESC

-- LIMIT 10000

In [0]:
%sql

SELECT 
    count_if(try_cast(value AS DOUBLE) = 0) as zero_pct,
    count_if(try_cast(value AS DOUBLE) < 1) as below_one,
    count_if(try_cast(value AS DOUBLE) > 100) as above_100

  FROM pep_dev.eval.own_stmt

WHERE prop = 'percentage' AND schema IN ('Ownership') 

-- HAVING count_if(try_cast(value AS DOUBLE) IS MAX() )
 


In [0]:
    SELECT
        dataset,
        COUNT(*)                     AS n_rows,
        MIN(try_cast(value AS DOUBLE))      AS min_pct,
        MAX(try_cast(value AS DOUBLE)  )      AS max_pct,
        count_if( try_cast(value AS DOUBLE) > 0 AND try_cast(value AS DOUBLE) < 1 )  AS below_one
    FROM   pep_dev.eval.own_stmt
    WHERE  prop = 'percentage' AND schema IN ('Ownership') 
    GROUP BY dataset
    ORDER BY max_pct 

In [0]:
        WITH step1 AS (
            SELECT entity_id, dataset, value,
                replace(value, '%', '') AS pct_text
            FROM   pep_dev.eval.own_stmt
            WHERE  prop = 'percentage' AND schema = 'Ownership'
        ), step2 as (

        SELECT entity_id,value, pct_text,
            CASE WHEN contains(pct_text, '–')
                    THEN try_cast(split_part(pct_text, '–', 1) AS DOUBLE)
                    ELSE try_cast(pct_text AS DOUBLE)
            END AS pct_lo,

            CASE WHEN contains(pct_text, '–')
                    THEN try_cast(split_part(pct_text, '–', 2) AS DOUBLE)
                    ELSE try_cast(pct_text AS DOUBLE)
            END AS pct_hi,dataset
            FROM step1),
            step3 as (
                SELECT entity_id,pct_text, 
                       CASE 
                             WHEN dataset = 'c4ads_xinjiang'
                             THEN pct_hi*100 
                             ELSE pct_hi
                             END as pct_hi ,

                       CASE 
                             WHEN dataset = 'c4ads_xinjiang'
                             THEN pct_lo*100 
                             ELSE pct_lo
                             END as pct_lo,
                             dataset

                FROM step2

                

                            
            ),

            step4 as (

                SELECT entity_id,pct_text,
                    CASE 
                            WHEN pct_hi=0 OR pct_hi>100 OR pct_hi IS NULL
                            THEN NULL                         
                            ELSE pct_hi 
                            END as pct_hi ,

                    CASE 
                            WHEN pct_lo=0 OR pct_lo>100 OR pct_lo IS NULL
                            THEN NULL
                            ELSE pct_lo  
                            END as pct_lo
                            
                FROM step3
                

            ),
            FINAL AS (

            SELECT entity_id,MIN(pct_lo) as  pct_min, MAX(pct_hi) as pct_max  FROM step4
            GROUP BY entity_id
            )

SELECT entity_id, pct_text, pct_lo, pct_hi
FROM step3
WHERE dataset = 'c4ads_xinjiang'
LIMIT 10            
            

        

In [0]:
SELECT COUNT(DISTINCT entity_id) FROM pep_dev.eval.own_stmt
WHERE schema = 'Ownership' AND prop = 'percentage' 

In [0]:
DESCRIBE pep_dev.bronze.statements

In [0]:
CREATE OR REPLACE TABLE pep_dev.eval.entity_ids AS
WITH layer1 AS (
    SELECT entity_id,
           collect_set(schema) AS schemas,
           coalesce(MAX(CASE WHEN canonical_id <> entity_id THEN canonical_id END),
                    MAX(canonical_id)) AS canonical_id
    FROM   pep_dev.bronze.statements
    WHERE  _snapshot_version = (SELECT MAX(snapshot_version) FROM pep_dev.bronze.snapshot_version_control)
    GROUP BY entity_id
)
SELECT entity_id, canonical_id,
       CASE WHEN array_contains(schemas, 'Person')  THEN 'Person'
            WHEN array_contains(schemas, 'Company') THEN 'Company'
            WHEN array_contains(schemas, 'Organization') THEN 'Organization'
            WHEN array_contains(schemas, 'LegalEntity') THEN 'LegalEntity'
            ELSE element_at(schemas, 1)
       END AS schema
FROM layer1

In [0]:
SELECT * FROM pep_dev.eval.entity_ids   

In [0]:
WITH ends 
as (
    SELECT entity_id as ownership_id,
           prop, 
           value as target_id
    FROM pep_dev.eval.own_stmt
    WHERE schema IN ('Ownership') AND  prop IN ('owner','asset')
),
canon as (
    SELECT DISTINCT canonical_id 
    FROM pep_dev.eval.entity_ids

)

SELECT src.prop,
       COUNT(*)                                                          AS total_ends,
       count_if(tgt.entity_id IS NOT NULL)                               AS matched_entity_id,
       count_if(tgt.entity_id IS NULL AND c.canonical_id IS NOT NULL)    AS rescued_by_canonical,
       count_if(tgt.entity_id IS NULL AND c.canonical_id IS NULL)        AS still_dangling
       
FROM ends as src 
LEFT JOIN pep_dev.eval.entity_ids as tgt
ON src.target_id = tgt.entity_id
LEFT JOIN canon as c
ON src.target_id=c.canonical_id
-- WHERE  tgt.entity_id IS NULL AND target_id LIKE  ('N%') 
GROUP BY prop
-- LIMIT 30

In [0]:
SELECT count(DISTINCT canonical_id) FROM pep_dev.bronze.statements
WHERE canonical_id<>entity_id
AND _snapshot_version = (SELECT MAX(snapshot_version) FROM pep_dev.bronze.snapshot_version_control)
-- AND canonical_id IS NOT NULL


In [0]:
    SELECT dataset,count(DISTINCT entity_id),
    COUNT(DISTINCT CASE WHEN prop='percentage' THEN entity_id END ) as pct,
    COUNT(DISTINCT CASE WHEN prop='endDate' THEN entity_id END ) as n_records_ended
    FROM 
    pep_dev.eval.own_stmt
    WHERE SCHEMA='Ownership'
    group by dataset

In [0]:
-- Q6 

WITH ends as 
(
    SELECT entity_id as ownership_id,
           prop, 
           value as target_id FROM pep_dev.eval.own_stmt
    WHERE prop IN ('owner','asset') 
),
resolved as (

        SELECT coalesce(s.canonical_id,e.target_id) as resolved_id,
               ownership_id,
               prop
        FROM ends as e
        LEFT JOIN pep_dev.eval.entity_ids as s 
        ON e.target_id = s.entity_id
),
edges as (
SELECT ownership_id,
        MAX(CASE WHEN prop = 'owner' THEN resolved_id END) as owner_id,
        MAX(CASE WHEN prop = 'asset' THEN resolved_id END ) as asset_id 
FROM 
resolved
GROUP BY ownership_id
),
list_A as (

   SELECT  DISTINCT owner_id FROM edges 
),

list_B (
    SELECT  DISTINCT asset_id FROM edges
),
middle_layer as( 
    SELECT count(owner_id)
    FROM list_A as a
    JOIN list_B as b 
    ON a.owner_id=b.asset_id
)

-- SELECT * FROM middle_layer

-- SELECT count(*)
-- FROM edges as e1 
-- JOIN edges as e2
-- ON  e1.asset_id=e2.owner_id
-- JOIN edges as e3 
-- ON e2.asset_id = e3.owner_id



-- SELECT count(*) FROM edges
-- where owner_id = asset_id;


    SELECT count(*)
    FROM edges e1
    JOIN edges e2 ON e1.asset_id = e2.owner_id
    WHERE e1.owner_id = e2.asset_id



--    SELECT DISTINCT asset_id FROM edges



In [0]:
-- Company property coverage: share of companies (distinct canonical_id) that have each property.
-- Decides which columns go into silver.legal_entity and which identifiers Phase 3 matching can rely on.
    
SELECT count(DISTINCT canonical_id) AS n_companies,
ROUND(100*n_companies/t.total,1) as coverage_pct,
prop
FROM
pep_dev.eval.own_stmt

CROSS JOIN (SELECT COUNT(DISTINCT canonical_id) as total FROM pep_dev.eval.own_stmt
    WHERE schema IN ('Company', 'Organization', 'LegalEntity')) as t  
WHERE SCHEMA IN ('Company', 'Organization', 'LegalEntity')
GROUP BY prop,t.total
ORDER BY n_companies DESC




In [0]:

    

SELECT value, count(DISTINCT canonical_id) as n_associated_entities,prop
FROM pep_dev.eval.own_stmt
WHERE SCHEMA IN ('Company', 'Organization', 'LegalEntity') AND prop IN ('topics') 
GROUP BY value,prop
ORDER BY n_associated_entities DESC
